# Pyr.ai Neuroglancer Viewer

This notebook creates Neuroglancer/Spelunker views for a selected CA3 root in the `zheng_ca3` datastack.

It:
- loads synapse data using the cache-first synapse artifact workflow;
- builds a mesh-only viewer for the selected neuron;
- visualizes all afferent and efferent synapses as annotation layers;
- generates a shortened Spelunker link using NGLui/CAVE state storage for reliable notebook/browser use.

If a valid local synapse table is available, this notebook uses the cached artifact. If no local artifact exists, it automatically queries CAVE, saves a validated parquet/metadata pair, and continues. Existing partial or invalid artifacts are not overwritten automatically.

In [1]:
root_id = 648518346450460332
datastack_name = "zheng_ca3"
materialization_version = 195
viewer_resolution = [18, 18, 45]
mesh_viewer_prefix = "https://spelunker.cave-explorer.org/"
image_source = "precomputed://gs://zheng_mouse_hippocampus_production/v2/img_aligned_sharded_18nm"
segmentation_source = "gs://zheng_mouse_hippocampus_production/v2/seg_m195/|neuroglancer-precomputed:"
segmentation_3d_opacity = 0.5
show_full_path = False


## CAVE Connection

Load the CAVE token through the shared auth helper without printing it, then connect to the `zheng_ca3` datastack.

In [2]:
import sys
from pathlib import Path

import caveclient
import numpy as np
from IPython.display import HTML, display
from neuroglancer import url_state

def discover_project_root(start=Path.cwd().resolve()):
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "helpers").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find the Pyr project root. Expected a parent directory containing "
        "notebooks/helpers."
    )


project_root = discover_project_root()
helpers_dir = project_root / "notebooks" / "helpers"
synapse_table_dir = project_root / "data" / "synapse_tables"

if str(helpers_dir) not in sys.path:
    sys.path.insert(0, str(helpers_dir))

from cave_auth import load_cave_token
from synapse_table_utils import artifact_paths, load_or_query_synapses, load_valid_synapse_artifact

token, cave_token_source = load_cave_token(project_root)

client = caveclient.CAVEclient(datastack_name, auth_token=token)


## Synapse Queries

Query afferent and efferent synapses at materialization 195. Coordinates are requested in the same voxel resolution used by the Pyr viewer state.

In [3]:
synapse_paths = artifact_paths(
    output_dir=synapse_table_dir,
    root_id=root_id,
    materialization_version=materialization_version,
)
cache_files_exist = synapse_paths["parquet_path"].exists() or synapse_paths["metadata_path"].exists()

if cache_files_exist:
    synapse_result = load_valid_synapse_artifact(
        output_dir=synapse_table_dir,
        root_id=root_id,
        datastack=datastack_name,
        materialization_version=materialization_version,
        desired_resolution=viewer_resolution,
    )
    if not synapse_result["ok"]:
        problem_text = "\n".join(f"- {problem}" for problem in synapse_result["problems"])
        raise RuntimeError(
            "Existing synapse artifact is missing, invalid, or incompatible with the current "
            f"root/materialization/resolution configuration:\n{problem_text}"
        )
else:
    synapse_result = load_or_query_synapses(
        client=client,
        output_dir=synapse_table_dir,
        root_id=root_id,
        datastack=datastack_name,
        materialization_version=materialization_version,
        desired_resolution=viewer_resolution,
        overwrite=False,
    )
    if not synapse_result["ok"]:
        problem_text = "\n".join(f"- {problem}" for problem in synapse_result["problems"])
        raise RuntimeError(f"Could not load or query synapse data:\n{problem_text}")

afferent_df = synapse_result["afferent_df"]
efferent_df = synapse_result["efferent_df"]
metadata = synapse_result["metadata"]

print(f"synapse source/status: {synapse_result['status']}")
print(f"afferent count: {len(afferent_df)}")
print(f"efferent count: {len(efferent_df)}")
print(f"total synapse rows: {metadata['total_synapse_rows']}")

if len(afferent_df) == 0 and len(efferent_df) == 0:
    raise RuntimeError(
        "No afferent or efferent synapses were available for this root; "
        "the viewer center cannot be computed from synapse coordinates."
    )


synapse source/status: loaded_cache
afferent count: 4807
efferent count: 29
total synapse rows: 4836


## Viewer Center

Use the seed neuron's synapse coordinates to center the view. Afferent rows contribute `post_pt_position`; efferent rows contribute `pre_pt_position`.

In [4]:
def position_array(df, column):
    if df.empty:
        return np.empty((0, 3), dtype=float)

    if column in df.columns:
        values = df[column].dropna().to_list()
        if values:
            return np.asarray(values, dtype=float)

    split_columns = [f"{column}_{axis}" for axis in "xyz"]
    if all(col in df.columns for col in split_columns):
        return df[split_columns].dropna().to_numpy(dtype=float)

    return np.empty((0, 3), dtype=float)


center_candidates = [
    position_array(afferent_df, "post_pt_position"),
    position_array(efferent_df, "pre_pt_position"),
]
center_candidates = [points for points in center_candidates if len(points) > 0]

if not center_candidates:
    raise RuntimeError("No afferent post or efferent pre positions were available to center the view.")

viewer_center = np.median(np.vstack(center_candidates), axis=0).round().astype(int).tolist()

## Direct Pyr State

Build a plain Python dictionary that mirrors the known-working public Pyr state. The segmentation source structure is kept exactly as authored here.

In [5]:
state = {
    "dimensions": {
        "x": [1.8e-8, "m"],
        "y": [1.8e-8, "m"],
        "z": [4.5e-8, "m"],
    },
    "position": viewer_center,
    "showSlices": False,
    "layers": [
        {
            "type": "image",
            "source": image_source,
            "name": "EM",
        },
        {
            "type": "segmentation",
            "source": {
                "url": segmentation_source,
                "subsources": {
                    "default": True,
                    "bounds": True,
                    "mesh": True,
                },
                "enableDefaultSubsources": False,
            },
            "segments": [str(root_id)],
            "selectedAlpha": 0.53,
            "name": "CA3 neuron",
        },
    ],
    "selectedLayer": {
        "layer": "CA3 neuron",
    },
    "layout": "xy-3d",
}


## Compact Diagnostic

Before encoding the URL, inspect the key fields and assert that the segmentation source points at `seg_m195` without Graphene or middleauth.

In [6]:
segmentation_layer = next(layer for layer in state["layers"] if layer["name"] == "CA3 neuron")
segmentation_source_url = segmentation_layer["source"]["url"]

print("state position:", state["position"])
print("layer names:", [layer["name"] for layer in state["layers"]])
print("segmentation source URL:", segmentation_source_url)
print("selected segment ID:", segmentation_layer["segments"][0])

assert "seg_m195" in segmentation_source_url
assert "graphene" not in segmentation_source_url.lower()
assert "middleauth" not in segmentation_source_url.lower()

state position: [56534, 70068, 1226]
layer names: ['EM', 'CA3 neuron']
segmentation source URL: gs://zheng_mouse_hippocampus_production/v2/seg_m195/|neuroglancer-precomputed:
selected segment ID: 648518346450460332


## Clickable Pyr Link

Encode the raw state dictionary into an inline Neuroglancer URL for Spelunker. The full URL is not printed.

In [7]:
ng_url = url_state.to_url(state, prefix=mesh_viewer_prefix)
display(HTML(f'<a href="{ng_url}" target="_blank" rel="noopener noreferrer">Open mesh-only view in Spelunker</a>'))


## Annotation Serialization Helper

Serialize point coordinates as Neuroglancer point annotations.

In [8]:
from neuroglancer import viewer_state
from neuroglancer.json_wrappers import to_json


def point_annotations(points, prefix):
    return [
        to_json(
            viewer_state.PointAnnotation(
                point=np.asarray(point, dtype=float).tolist(),
                id=f"{prefix}-{index:03d}",
            )
        )
        for index, point in enumerate(points)
    ]


## Full-Synapse Spelunker View

Build a full afferent/efferent synapse annotation view from the existing raw Pyr state and generate a shortened Spelunker link through NGLui/CAVE state storage.

In [10]:
# Build a Spelunker viewer URL from the existing annotations without subsampling.

import copy
import warnings
from nglui.statebuilder import ViewerState, neuroglancer_url

spelunker_url = neuroglancer_url(target_site="spelunker")
all_afferent_points = position_array(afferent_df, "post_pt_position")
all_efferent_points = position_array(efferent_df, "pre_pt_position")

if len(all_afferent_points) != len(afferent_df):
    raise RuntimeError(
        f"Could not extract one afferent point for each afferent row "
        f"({len(all_afferent_points)} points for {len(afferent_df)} rows)."
    )
if len(all_efferent_points) != len(efferent_df):
    raise RuntimeError(
        f"Could not extract one efferent point for each efferent row "
        f"({len(all_efferent_points)} points for {len(efferent_df)} rows)."
    )

full_synapse_state = copy.deepcopy(state)
for full_synapse_layer in full_synapse_state.get("layers", []):
    if full_synapse_layer.get("name") == "CA3 neuron" and full_synapse_layer.get("type") == "segmentation":
        full_synapse_layer["objectAlpha"] = segmentation_3d_opacity
full_synapse_local_annotation_source = {
    "url": "local://annotations",
    "transform": {
        "outputDimensions": copy.deepcopy(full_synapse_state["dimensions"]),
    },
}
full_synapse_state["layers"].extend(
    [
        {
            "type": "annotation",
            "source": copy.deepcopy(full_synapse_local_annotation_source),
            "annotations": point_annotations(all_afferent_points, "afferent"),
            "annotationColor": "#00ffff",
            "name": "Afferent synapses",
        },
        {
            "type": "annotation",
            "source": copy.deepcopy(full_synapse_local_annotation_source),
            "annotations": point_annotations(all_efferent_points, "efferent"),
            "annotationColor": "#ff00ff",
            "name": "Efferent synapses",
        },
    ]
)

full_annotation_layers = {
    layer["name"]: layer
    for layer in full_synapse_state["layers"]
    if layer.get("type") == "annotation"
}
afferent_annotation_count = len(full_annotation_layers["Afferent synapses"]["annotations"])
efferent_annotation_count = len(full_annotation_layers["Efferent synapses"]["annotations"])

assert afferent_annotation_count == len(afferent_df)
assert efferent_annotation_count == len(efferent_df)

inline_spelunker_url = url_state.to_url(full_synapse_state, prefix=spelunker_url)
inline_url_length = len(inline_spelunker_url)

short_url = None
shortening_method = None
state_id = None
shortening_error = None

try:
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"No dimensions provided or inferred\. Using a null CoordSpace.*",
            category=UserWarning,
        )
        full_synapse_viewer = ViewerState(
            base_state=full_synapse_state,
            client=client,
            target_site="spelunker",
            infer_coordinates=False,
        )
        short_url = full_synapse_viewer.to_url(
            shorten=True,
            client=client,
            target_site="spelunker",
        )
    shortening_method = "nglui ViewerState.to_url(shorten=True, target_site='spelunker')"
except Exception as exc:
    shortening_error = f"NGLui shortening failed: {type(exc).__name__}: {exc}"

if short_url is None:
    try:
        state_id = client.state.upload_state_json(full_synapse_state)
        short_url = client.state.build_neuroglancer_url(
            state_id,
            target_site="spelunker",
        )
        shortening_method = "direct client.state upload + build_neuroglancer_url(target_site='spelunker')"
        shortening_error = None
    except Exception as exc:
        direct_error = f"Direct CAVE state shortening failed: {type(exc).__name__}: {exc}"
        shortening_error = f"{shortening_error}; {direct_error}" if shortening_error else direct_error

print(f"afferent annotation count: {afferent_annotation_count}")
print(f"efferent annotation count: {efferent_annotation_count}")
print(f"total annotation count: {afferent_annotation_count + efferent_annotation_count}")
print(f"inline Spelunker URL length: {inline_url_length}")

if short_url is None:
    raise RuntimeError(shortening_error)

print(f"shortening method: {shortening_method}")
if state_id is not None:
    print(f"state ID: {state_id}")
print(f"shortened URL length: {len(short_url)}")
print(f"shortened URL: {short_url}")
display(
    HTML(
        f'<a href="{short_url}" target="_blank" rel="noopener noreferrer">'
        'Open full synapse view in Spelunker'
        '</a>'
    )
)


afferent annotation count: 4807
efferent annotation count: 29
total annotation count: 4836
inline Spelunker URL length: 477474
shortening method: nglui ViewerState.to_url(shorten=True, target_site='spelunker')
shortened URL length: 109
shortened URL: https://spelunker.cave-explorer.org/#!middleauth+https://global.daf-apis.com/nglstate/api/v1/4532489876406272
